시간날 때 다음 예제를 직접 해보기

https://ryuzyproject.tistory.com/104

# 1. 이안류 CCTV 데이터셋

[AI허브 이안류 데이터](https://www.aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&aihubDataSe=data&dataSetSn=71297)

AI Hub의 '이안류 CCTV 데이터'는 우리나라 주요 해수욕장(해운대, 송정, 대천, 중문, 낙산)에서 이안류 발생 여부와 위치를 모니터링하기 위해 구축된 인공지능 학습용 데이터셋입니다.

해수욕장 주변에 설치된 CCTV 영상을 이미지로 변환하여, 이안류 발생 여부와 위치를 가시화하는 모델 개발에 활용할 수 있습니다.

이 데이터셋은 이안류 탐지 및 예측 시스템 개발에 필수적인 자료를 제공하며, 해수욕객의 안전을 위한 응용 서비스 구성에 활용될 수 있습니다.

이안류는 해안에서 먼 바다로 빠르게 이동하는 폭이 좁은 바닷물의 흐름으로, 기상 상태가 양호한 경우에도 나타나며, 얕은 곳에 있던 해수욕객을 순식간에 수심이 깊은 먼 바다로 이동시켜 인명사고를 유발할 수 있습니다.

따라서 이러한 데이터셋은 해수욕장 안전 관리 및 이안류 예측 모델 개발에 중요한 역할을 합니다.

# 2. YOLO 데이터셋 만들기
### 1. JSON Bounding Box > YOLO Bounding Box
- 원본 JSON의 drawing에는 객체를 둘러싼 사각형의 점 좌표가 들어있음
- YOLO의 일반적인 객체 탐지 라벨 형식
    - `클래스id`, `x센터`, `y센터`, `너비`, `높이`
    - 네개의 좌표를 모두 0 ~ 1 사이로 정규화

### 2. 원본 이미지와 JSON 라벨을 연결한 뒤 train/val/test 폴더로 나눔
- 같은 원본 영상에서 나온 프레임이 train과 test에 동시에 들어가면 성능이 실제보다 높게 보일 수 있음
- 파일명 앞부분을 그룹으로 사용해 가능한 한 같은 영상 계열이 한 ㅅ트에만 들어가도록 분리
- 데이터 수가 적으면 YOLO 성능이 낮아질 수 있음 (전체 데이터셋을 가지고 써보면 좋겠다는 의미)
    - 샘플 데이터에는 전체 이미지가 360개밖에 없어서
        - 전체 이미지: 360
        - 이안류 객체가 있는 이미지: 240장
        - 배경이미지: 120장
- YOLO는 단순히 이미지를 외우는 것이 아니라, 여러 이미지에서 반복적으로 시각적 특징을 학습하여 새로운 이미지에서도 객체를 찾아야함.

> YOLO는 단순히 이미지를 외우는 것이 아니라, 여러 이미지에서 반복적으로 나타나는 시각적 특징을 학습하여 새로운 이미지에서도 객체를 찾아야 함. 단숞한 파일 개수가 아니라 서로 다른 상황을 얼마나 다양하게 포함하고 있는지가 중요

- CCTV 연속 프레임에서는 무작위로 이미지를 섞어 나누는 방법을 사용하지 않음
    - 서로 몇초 차이인 것의 동일한 프레임이 Train과 Test를 동시에 들어가면 모델은 Test 이미지를 처음 보는 것이 아니라, 학습 때 본 장면과 매우 유사한 장면을 다시 보는 것과 비슷해짐 (데이터 누수)
- 전체 데이터셋을 확보할 수 있다면 아래 단위로 분리하는 것이 좋음
    - CCTV 카메라 각도
    - 해수욕장 위치
    - 날짜
    - 촬영 시간대


# **3. Detection이 문제 정의에 적합한지 검토**
> 이안류 영역을 Bounding Box로 감싸는 ObjectDetection 방식은 이안류처럼 형태가 불규칙하고 경계가 명확하지 않은 영역일 경우 Segmentation이 더 적합할 수 있음